# LeetCode 363 — Max Sum of Rectangle No Larger Than K bằng Python

Notebook này giảng **từ căn bản** một bài **Hard** kinh điển về ma trận. Bài này là chỗ gặp nhau của ba ý tưởng rất đáng học: **prefix sum** (tổng tiền tố), **nén 2 chiều thành 1 chiều**, và **tìm kiếm nhị phân trên danh sách đã sắp xếp**.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxSumSubmatrix(self, matrix: List[List[int]], k: int) -> int:

```

## Sau notebook này, bạn sẽ nắm được

1. Đọc đúng đề: "hình chữ nhật con" trong ma trận là gì, có bao nhiêu hình, và "tổng không vượt quá `k`" nghĩa là gì.
2. Công cụ nền tảng, dạy lại từ đầu: ma trận dạng `list` lồng `list`, **prefix sum 1 chiều**, **prefix sum 2 chiều** (công thức bao hàm – loại trừ), **binary search** với `bisect_left` / `insort`, thuật toán **Kadane**, chuyển vị ma trận bằng `zip(*matrix)`.
3. Bốn bậc lời giải, đi từ trực giác lên tối ưu:
   - **Cách 0:** Brute force ngây thơ — thử mọi hình chữ nhật, cộng tay từng ô: `O(m³·n³)`.
   - **Cách 1:** Brute force + **prefix sum 2D** — mỗi hình chữ nhật tính tổng trong `O(1)`: `O(m²·n²)`.
   - **Cách 2 (cách chính):** Cố định cặp hàng `(top, bottom)`, **nén các cột** thành một mảng 1 chiều, rồi giải bài 1D "tổng đoạn con lớn nhất `<= k`" bằng **prefix sum + list đã sắp xếp + `bisect_left`**: `O(m²·n log n)` phép tìm kiếm (nói rõ cái giá của `insort` ở mục 12).
   - **Cách 3 (tối ưu thực tế):** Thêm **Kadane** để bỏ qua bước tìm kiếm khi có thể, **dừng sớm** khi đạt đúng `k`, và **cố định chiều nhỏ hơn** (trả lời câu Follow-up của đề).
4. Trace (mô phỏng) từng bước bài 1D thật kỹ, rồi trace cả thuật toán 2D.
5. Vì sao đúng (**invariant** — "bất biến", điều luôn đúng sau mỗi bước), độ phức tạp, lỗi hay gặp, mẹo nhận dạng, bài liên quan và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Given an `m x n` matrix `matrix` and an integer `k`, return *the max sum of a rectangle in the matrix such that its sum is no larger than* `k`.

It is **guaranteed** that there will be a rectangle with a sum no larger than `k`.

**Example 1**

```text
Input: matrix = [[1,0,1],[0,-2,3]], k = 2
Output: 2
Explanation: Because the sum of the blue rectangle [[0, 1], [-2, 3]] is 2,
and 2 is the max number no larger than k (k = 2).
```

**Example 2**

```text
Input: matrix = [[2,2,-1]], k = 3
Output: 3
```

**Constraints**

- `m == matrix.length`
- `n == matrix[i].length`
- `1 <= m, n <= 100`
- `-100 <= matrix[i][j] <= 100`
- `-10^5 <= k <= 10^5`

**Follow up:** What if the number of rows is much larger than the number of columns?

### Dịch nhanh

- Cho ma trận số nguyên `matrix` có `m` hàng, `n` cột, và một số nguyên `k`.
- Một **hình chữ nhật** (rectangle, còn gọi là **ma trận con** — submatrix) là một khối ô **liền nhau**, chọn bởi một đoạn hàng liên tiếp và một đoạn cột liên tiếp.
- Trong tất cả các hình chữ nhật có **tổng các ô `<= k`** ("no larger than k" = không lớn hơn `k`), hãy trả về **tổng lớn nhất**.
- Đề **bảo đảm** luôn có ít nhất một hình chữ nhật thỏa — nên ta không phải lo trường hợp "không có đáp án".
- Lưu ý từ constraints: ô có thể **âm**, `k` có thể **âm** ⇒ đáp án cũng có thể là số **âm**. Hình chữ nhật phải có **ít nhất 1 ô** (không có "hình rỗng tổng 0").
- **Follow up** hỏi: nếu số hàng lớn hơn rất nhiều so với số cột thì làm sao? Ta sẽ trả lời ở Cách 3.

## 2. Hiểu đề bằng tay

### Một hình chữ nhật được xác định bởi 4 con số

Chọn hàng trên cùng `top`, hàng dưới cùng `bottom` (`top <= bottom`), cột trái `left`, cột phải `right` (`left <= right`). Hình chữ nhật gồm mọi ô `matrix[r][c]` với `top <= r <= bottom` và `left <= c <= right`.

Ví dụ 1: `matrix = [[1,0,1],[0,-2,3]]` — viết thành lưới:

| | cột 0 | cột 1 | cột 2 |
|---|---:|---:|---:|
| **hàng 0** | 1 | 0 | 1 |
| **hàng 1** | 0 | -2 | 3 |

Hình chữ nhật "màu xanh" của đề là `top = 0, bottom = 1, left = 1, right = 2`:

```text
 0   1
-2   3      tổng = 0 + 1 + (-2) + 3 = 2  <= k = 2  ✔
```

### Có bao nhiêu hình chữ nhật?

- Số cách chọn cặp hàng `top <= bottom` trong `m` hàng là `m·(m+1)/2` (chọn 2 hàng khác nhau, hoặc 1 hàng dùng cho cả hai).
- Tương tự, số cặp cột là `n·(n+1)/2`.
- Tổng số hình chữ nhật = `m(m+1)/2 · n(n+1)/2`.

Ví dụ 1: `m = 2, n = 3` ⇒ `3 · 6 = 18` hình. Ô code dưới liệt kê **cả 18** hình, tính tổng và đánh dấu những hình có tổng `<= k`.

In [1]:
from typing import List   # LeetCode tự import sẵn; trong notebook ta phải tự import
import bisect
import random
import time


def rectangles_report(matrix, k):
    """In MỌI hình chữ nhật (top, bottom, left, right) với tổng, và chọn tổng lớn nhất <= k."""
    m, n = len(matrix), len(matrix[0])
    best = None
    for top in range(m):
        for bottom in range(top, m):
            for left in range(n):
                for right in range(left, n):
                    s = sum(matrix[r][c] for r in range(top, bottom + 1) for c in range(left, right + 1))
                    ok = s <= k
                    if ok and (best is None or s > best):
                        best = s
                    print(f"hàng {top}..{bottom}, cột {left}..{right}: tổng = {s:>2} {'<= k' if ok else '>  k'}")
    print("Đáp án (tổng lớn nhất <= k):", best)


rectangles_report([[1, 0, 1], [0, -2, 3]], k=2)

hàng 0..0, cột 0..0: tổng =  1 <= k
hàng 0..0, cột 0..1: tổng =  1 <= k
hàng 0..0, cột 0..2: tổng =  2 <= k
hàng 0..0, cột 1..1: tổng =  0 <= k
hàng 0..0, cột 1..2: tổng =  1 <= k
hàng 0..0, cột 2..2: tổng =  1 <= k
hàng 0..1, cột 0..0: tổng =  1 <= k
hàng 0..1, cột 0..1: tổng = -1 <= k
hàng 0..1, cột 0..2: tổng =  3 >  k
hàng 0..1, cột 1..1: tổng = -2 <= k
hàng 0..1, cột 1..2: tổng =  2 <= k
hàng 0..1, cột 2..2: tổng =  4 >  k
hàng 1..1, cột 0..0: tổng =  0 <= k
hàng 1..1, cột 0..1: tổng = -2 <= k
hàng 1..1, cột 0..2: tổng =  1 <= k
hàng 1..1, cột 1..1: tổng = -2 <= k
hàng 1..1, cột 1..2: tổng =  1 <= k
hàng 1..1, cột 2..2: tổng =  3 >  k
Đáp án (tổng lớn nhất <= k): 2


### Nhận xét từ bảng trên

- Có **hai** hình đạt tổng `2`: hàng `0..0`, cột `0..2` (tức `[1, 0, 1]`) và hàng `0..1`, cột `1..2` (hình màu xanh của đề). Đề chỉ hỏi **giá trị tổng**, nên chọn hình nào cũng được.
- Hình lớn nhất (cả ma trận) có tổng `3 > 2` ⇒ **không** được chọn. Đây là điểm khác bài "tổng lớn nhất" thông thường: tổng lớn nhất **không** phải lúc nào cũng là đáp án, vì có trần `k`.
- Tổng có thể **giảm** khi mở rộng hình (do có số âm). Vì vậy "mở rộng thêm thì tổng tăng" là **sai** — chi tiết này sẽ giết chết ý tưởng cửa sổ trượt (xem phần Lỗi thường gặp).

### Ví dụ 2 và vài edge case nên nhớ ngay từ đầu

| `matrix` | `k` | Đáp án | Vì sao |
|---|---:|---:|---|
| `[[2,2,-1]]` | 3 | 3 | `[2, 2, -1]` tổng `3` (đúng bằng `k`) |
| `[[5]]` | 5 | 5 | chỉ một ô, `5 <= 5` |
| `[[-3]]` | 0 | -3 | chỉ một hình; đáp án **âm** — không được trả `0` |
| `[[-5,-2]]` | -1 | -2 | `k` âm; các tổng là `-5, -2, -7` ⇒ lớn nhất `<= -1` là `-2` |
| `[[2],[-1],[3]]` | 3 | 3 | ma trận 1 cột; các tổng `2, 1, 4, -1, 2, 3` — cả cột là `4 > 3`, lớn nhất `<= 3` là `[3]` |

In [2]:
def brute_answer(matrix, k):
    """Đáp án bằng cách thử hết (dùng để kiểm tra các ví dụ nhỏ)."""
    m, n = len(matrix), len(matrix[0])
    sums = [sum(matrix[r][c] for r in range(t, b + 1) for c in range(l, rr + 1))
            for t in range(m) for b in range(t, m) for l in range(n) for rr in range(l, n)]
    return max(s for s in sums if s <= k)


for mat, k in [([[2, 2, -1]], 3), ([[5]], 5), ([[-3]], 0), ([[-5, -2]], -1), ([[2], [-1], [3]], 3)]:
    print(f"matrix = {mat}, k = {k} -> {brute_answer(mat, k)}")

matrix = [[2, 2, -1]], k = 3 -> 3
matrix = [[5]], k = 5 -> 5
matrix = [[-3]], k = 0 -> -3
matrix = [[-5, -2]], k = -1 -> -2
matrix = [[2], [-1], [3]], k = 3 -> 3


## 3. Nền tảng 1: ma trận trong Python

Python không có kiểu "ma trận" riêng; ta dùng **list lồng list**: một `list` các hàng, mỗi hàng là một `list` các số.

- `len(matrix)` = số **hàng** `m`; `len(matrix[0])` = số **cột** `n` (đề bảo đảm có ít nhất 1 hàng và 1 cột).
- `matrix[r]` là **cả hàng** `r`; `matrix[r][c]` là ô ở hàng `r`, cột `c` (chỉ số bắt đầu từ `0`).
- Lấy **một cột** thì không có cú pháp sẵn: phải gom `matrix[r][c]` của mọi hàng `r`.
- Tạo ma trận toàn số 0 kích thước `R x C`: **`[[0] * C for _ in range(R)]`**. Đừng viết `[[0] * C] * R` — cách đó tạo `R` tham chiếu tới **cùng một hàng**, sửa một hàng là sửa tất cả (ô code dưới cho thấy).

In [3]:
matrix = [[1, 0, 1],
          [0, -2, 3]]
print("m =", len(matrix), "| n =", len(matrix[0]))
print("hàng 1:", matrix[1], "| ô (1, 2):", matrix[1][2])
print("cột 1:", [matrix[r][1] for r in range(len(matrix))])

good = [[0] * 3 for _ in range(2)]
bad = [[0] * 3] * 2
good[0][0] = 9
bad[0][0] = 9
print("tạo đúng:", good)    # chỉ hàng 0 đổi
print("tạo sai :", bad)     # cả hai hàng cùng đổi!

m = 2 | n = 3
hàng 1: [0, -2, 3] | ô (1, 2): 3
cột 1: [0, -2]
tạo đúng: [[9, 0, 0], [0, 0, 0]]
tạo sai : [[9, 0, 0], [9, 0, 0]]


## 4. Nền tảng 2: prefix sum 1 chiều (tổng tiền tố)

**Câu hỏi:** cho mảng `a`, cần tính tổng đoạn `a[l..r]` (gồm cả hai đầu) **rất nhiều lần**. Cộng tay mỗi lần tốn `O(r - l + 1)`. Có cách nào trả lời trong `O(1)`?

**Ý tưởng:** tính trước mảng **prefix** `P` có độ dài `len(a) + 1`:

- `P[0] = 0` (tổng của **0** phần tử đầu tiên),
- `P[i + 1] = P[i] + a[i]` ⇒ `P[i]` = tổng của **`i` phần tử đầu tiên** `a[0] + ... + a[i-1]`.

Khi đó:

$$\text{sum}(a[l..r]) = P[r+1] - P[l]$$

vì `P[r+1]` là tổng `a[0..r]`, `P[l]` là tổng `a[0..l-1]`, trừ đi thì còn đúng `a[l..r]`.

**Ví dụ số:** `a = [5, -6, 3, -2, 4]`

| `i` | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---:|---:|---:|---:|---:|---:|
| `P[i]` | 0 | 5 | -1 | 2 | 0 | 4 |

Tổng `a[2..4] = 3 + (-2) + 4 = 5`, và `P[5] - P[2] = 4 - (-1) = 5`. ✔

Vì sao lại cần `P[0] = 0`? Để đoạn **bắt đầu từ chỉ số 0** cũng dùng được công thức: `sum(a[0..r]) = P[r+1] - P[0]`. Chi tiết nhỏ này sẽ quay lại rất quan trọng trong bài 1D.

In [4]:
def prefix_1d(a):
    P = [0] * (len(a) + 1)
    for i, x in enumerate(a):
        P[i + 1] = P[i] + x
    return P


a = [5, -6, 3, -2, 4]
P = prefix_1d(a)
print("P =", P)
for l, r in [(2, 4), (0, 1), (1, 3), (4, 4)]:
    print(f"sum(a[{l}..{r}]) = {sum(a[l:r + 1]):>2}  |  P[{r + 1}] - P[{l}] = {P[r + 1]} - ({P[l]}) = {P[r + 1] - P[l]}")

P = [0, 5, -1, 2, 0, 4]
sum(a[2..4]) =  5  |  P[5] - P[2] = 4 - (-1) = 5
sum(a[0..1]) = -1  |  P[2] - P[0] = -1 - (0) = -1
sum(a[1..3]) = -5  |  P[4] - P[1] = 0 - (5) = -5
sum(a[4..4]) =  4  |  P[5] - P[4] = 4 - (0) = 4


## 5. Cách 0 — Brute force ngây thơ: thử mọi hình, cộng tay từng ô

Cách đầu tiên ai cũng nghĩ ra: 4 vòng `for` chọn `(top, bottom, left, right)`, rồi 2 vòng `for` nữa cộng các ô bên trong. Giữ lại tổng lớn nhất mà `<= k`.

Chú ý khởi tạo `best = float("-inf")` (âm vô cùng), **không** phải `0`: đáp án có thể âm (ví dụ `[[-3]], k = 0 → -3`).

**Chi phí:** có khoảng `m²n²/4` hình, mỗi hình cộng tới `m·n` ô ⇒ `O(m³·n³)`. Với `m = n = 100` là cỡ `10^11` phép cộng — không thể chạy nổi. Nhưng nó **rất dễ viết đúng**, nên ta giữ lại làm "trọng tài" để kiểm tra các cách nhanh hơn.

In [5]:
def max_sum_naive(matrix, k):
    m, n = len(matrix), len(matrix[0])
    best = float("-inf")
    for top in range(m):
        for bottom in range(top, m):
            for left in range(n):
                for right in range(left, n):
                    s = 0
                    for r in range(top, bottom + 1):          # cộng tay từng ô
                        for c in range(left, right + 1):
                            s += matrix[r][c]
                    if s <= k and s > best:
                        best = s
    return best


print(max_sum_naive([[1, 0, 1], [0, -2, 3]], 2))   # 2
print(max_sum_naive([[2, 2, -1]], 3))              # 3
print(max_sum_naive([[-5, -2]], -1))               # -2

2
3
-2


### Nút thắt của Cách 0

Hai hình chữ nhật cạnh nhau, ví dụ cột `0..1` và cột `0..2` (cùng hàng), chung gần hết các ô — nhưng Cách 0 cộng lại từ đầu. **Công việc bị lặp** chính là phần "cộng tay từng ô".

Ở mảng 1 chiều, ta đã biết cách xóa sự lặp này: **prefix sum**. Câu hỏi tự nhiên: có **prefix sum 2 chiều** không? Có — và đó là nền tảng tiếp theo.

## 6. Nền tảng 3: prefix sum 2 chiều

### Định nghĩa

Tạo bảng `P` kích thước `(m + 1) x (n + 1)`, trong đó:

> `P[r][c]` = tổng mọi ô của hình chữ nhật **góc trên-trái `(0, 0)`**, gồm `r` hàng đầu và `c` cột đầu (tức các ô `matrix[i][j]` với `i < r`, `j < c`).

Hàng `0` và cột `0` của `P` toàn số `0` (hình có 0 hàng hoặc 0 cột) — giống `P[0] = 0` ở bản 1D.

### Công thức xây bảng — bao hàm – loại trừ (inclusion–exclusion)

Muốn tính `P[r+1][c+1]` (hình từ `(0,0)` tới ô `(r, c)`), ghép từ các mảnh đã có:

```text
P[r+1][c+1] =   P[r][c+1]        (phần phía TRÊN: r hàng đầu, c+1 cột đầu)
              + P[r+1][c]        (phần bên TRÁI: r+1 hàng đầu, c cột đầu)
              - P[r][c]          (góc trên-trái bị cộng HAI lần -> trừ bớt một lần)
              + matrix[r][c]     (chính ô (r, c))
```

Hình dung: "trên" và "trái" chồng lên nhau ở góc `P[r][c]`; cộng cả hai thì góc đó bị đếm hai lần, nên phải trừ đi một lần. Đó là ý nghĩa của **bao hàm – loại trừ**.

### Ví dụ số với Ví dụ 1

`matrix = [[1,0,1],[0,-2,3]]` ⇒ bảng `P` (3 hàng x 4 cột):

| `P` | c = 0 | c = 1 | c = 2 | c = 3 |
|---|---:|---:|---:|---:|
| **r = 0** | 0 | 0 | 0 | 0 |
| **r = 1** | 0 | 1 | 1 | 2 |
| **r = 2** | 0 | 1 | -1 | 3 |

Kiểm tra ô `P[2][3]` (cả ma trận): `P[1][3] + P[2][2] - P[1][2] + matrix[1][2] = 2 + (-1) - 1 + 3 = 3`. Và đúng là `1+0+1+0-2+3 = 3`. ✔

In [6]:
def build_prefix_2d(matrix):
    m, n = len(matrix), len(matrix[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for r in range(m):
        for c in range(n):
            P[r + 1][c + 1] = P[r][c + 1] + P[r + 1][c] - P[r][c] + matrix[r][c]
    return P


P2 = build_prefix_2d([[1, 0, 1], [0, -2, 3]])
for r, row in enumerate(P2):
    print(f"P[{r}] =", row)

P[0] = [0, 0, 0, 0]
P[1] = [0, 1, 1, 2]
P[2] = [0, 1, -1, 3]


### Truy vấn tổng một hình chữ nhật bất kỳ trong `O(1)`

Hình chữ nhật hàng `r1..r2`, cột `c1..c2` (gồm cả hai đầu). Lại dùng bao hàm – loại trừ:

```text
sum = P[r2+1][c2+1]      (hình lớn từ (0,0) tới (r2,c2))
    - P[r1][c2+1]        (bỏ dải phía TRÊN hàng r1)
    - P[r2+1][c1]        (bỏ dải bên TRÁI cột c1)
    + P[r1][c1]          (góc trên-trái bị trừ HAI lần -> cộng lại một lần)
```

**Ví dụ số:** hình màu xanh `r1 = 0, r2 = 1, c1 = 1, c2 = 2`:

`P[2][3] - P[0][3] - P[2][1] + P[0][1] = 3 - 0 - 1 + 0 = 2`. ✔ (khớp `0 + 1 - 2 + 3 = 2`)

**Ví dụ số 2:** chỉ ô `(1, 1)`: `P[2][2] - P[1][2] - P[2][1] + P[1][1] = -1 - 1 - 1 + 1 = -2`. ✔

In [7]:
def rect_sum(P, r1, c1, r2, c2):
    """Tổng hình chữ nhật hàng r1..r2, cột c1..c2 (gồm cả hai đầu), dùng bảng prefix 2D."""
    return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]


M1 = [[1, 0, 1], [0, -2, 3]]
print("hình xanh:", rect_sum(P2, 0, 1, 1, 2))   # 2
print("ô (1,1):  ", rect_sum(P2, 1, 1, 1, 1))   # -2

# Kiểm tra với cộng tay trên MỌI hình chữ nhật của ma trận ngẫu nhiên
random.seed(363)
for _ in range(200):
    m, n = random.randint(1, 4), random.randint(1, 4)
    mat = [[random.randint(-9, 9) for _ in range(n)] for _ in range(m)]
    P = build_prefix_2d(mat)
    for r1 in range(m):
        for r2 in range(r1, m):
            for c1 in range(n):
                for c2 in range(c1, n):
                    direct = sum(mat[r][c] for r in range(r1, r2 + 1) for c in range(c1, c2 + 1))
                    assert rect_sum(P, r1, c1, r2, c2) == direct
print("rect_sum khớp cộng tay trên 200 ma trận ngẫu nhiên ✔")

hình xanh: 2
ô (1,1):   -2
rect_sum khớp cộng tay trên 200 ma trận ngẫu nhiên ✔


## 7. Cách 1 — Brute force + prefix sum 2D: `O(m²·n²)`

Giữ nguyên 4 vòng chọn hình, nhưng thay 2 vòng "cộng tay" bằng **một** lời gọi `rect_sum` `O(1)`.

- Xây bảng `P`: `O(m·n)`.
- Số hình: `m(m+1)/2 · n(n+1)/2` ⇒ thời gian `O(m²·n²)`; bộ nhớ `O(m·n)` cho bảng `P`.

Với `m = n = 100`: `5050 · 5050 ≈ 25,5 triệu` hình. Trong C++ có thể chạy kịp, nhưng trong Python mỗi phép tính tốn cỡ vài chục nano-giây ⇒ mất hàng chục giây ⇒ **Time Limit Exceeded (TLE)** (quá thời gian). Ta cần giảm thêm một bậc.

In [8]:
def max_sum_prefix2d(matrix, k):
    m, n = len(matrix), len(matrix[0])
    P = build_prefix_2d(matrix)
    best = float("-inf")
    for top in range(m):
        for bottom in range(top, m):
            for left in range(n):
                for right in range(left, n):
                    s = rect_sum(P, top, left, bottom, right)
                    if s <= k and s > best:
                        best = s
    return best


print(max_sum_prefix2d([[1, 0, 1], [0, -2, 3]], 2))   # 2
print(max_sum_prefix2d([[2, 2, -1]], 3))              # 3
print("số hình khi m = n = 100:", (100 * 101 // 2) ** 2)

2
3
số hình khi m = n = 100: 25502500


### Nút thắt của Cách 1

Nhìn lại 4 vòng lặp: với **mỗi** cặp hàng `(top, bottom)`, ta thử **mọi** cặp cột `(left, right)` — tốn `O(n²)`. Có `O(m²)` cặp hàng ⇒ `O(m²·n²)`.

Câu hỏi then chốt: **khi đã cố định `(top, bottom)`**, có cách nào tìm cặp cột tốt nhất **nhanh hơn `O(n²)`** không? Muốn trả lời, ta phải nhìn bài toán theo một góc khác.

## 8. Bước đột phá: cố định cặp hàng, **nén** mỗi cột thành một số

Cố định `top` và `bottom`. Mọi hình chữ nhật dùng đúng dải hàng này chỉ còn khác nhau ở `(left, right)`.

Với mỗi cột `c`, gộp các ô từ hàng `top` tới `bottom` thành **một con số**:

$$\text{col\_sum}[c] = \sum_{r=top}^{bottom} \text{matrix}[r][c]$$

Khi đó tổng hình chữ nhật `(top, bottom, left, right)` chỉ là:

$$\text{col\_sum}[left] + \dots + \text{col\_sum}[right]$$

tức là **tổng một đoạn con liên tiếp** của mảng 1 chiều `col_sum`!

**Ví dụ 1**, `top = 0, bottom = 1`:

```text
hàng 0:   1   0   1
hàng 1:   0  -2   3
          ----------   cộng dọc từng cột
col_sum:  1  -2   4
```

Hình màu xanh (cột `1..2`) có tổng `col_sum[1] + col_sum[2] = -2 + 4 = 2`. ✔

⇒ Bài 2D trở thành: **với mỗi cặp hàng, giải bài 1D "tổng đoạn con lớn nhất mà `<= k`"** trên mảng `col_sum` độ dài `n`. Lấy max qua mọi cặp hàng là xong.

Đây là kỹ thuật **"nén chiều"** (dimension reduction) — một mẹo cực kỳ phổ biến cho bài hình chữ nhật trong ma trận (gặp lại ở #85, #1074...).

In [9]:
def show_compression(matrix):
    m, n = len(matrix), len(matrix[0])
    for top in range(m):
        for bottom in range(top, m):
            col_sum = [sum(matrix[r][c] for r in range(top, bottom + 1)) for c in range(n)]
            print(f"top = {top}, bottom = {bottom}: col_sum = {col_sum}")


show_compression([[1, 0, 1], [0, -2, 3]])

top = 0, bottom = 0: col_sum = [1, 0, 1]
top = 0, bottom = 1: col_sum = [1, -2, 4]
top = 1, bottom = 1: col_sum = [0, -2, 3]


### Cập nhật `col_sum` tăng dần — không cộng lại từ đầu

Cố định `top`, rồi cho `bottom` chạy `top, top+1, ..., m-1`. Mỗi lần `bottom` tăng 1, dải hàng chỉ **thêm đúng một hàng** ⇒

```python
col_sum[c] += matrix[bottom][c]     # với mọi c
```

Chỉ tốn `O(n)` mỗi bước thay vì cộng lại cả dải. (Đây chính là "prefix sum theo chiều dọc", làm dần dần.)

Ví dụ 1 với `top = 0`: bắt đầu `col_sum = [0, 0, 0]`
→ `bottom = 0`: cộng hàng 0 ⇒ `[1, 0, 1]`
→ `bottom = 1`: cộng hàng 1 ⇒ `[1, -2, 4]`.

In [10]:
M1 = [[1, 0, 1], [0, -2, 3]]
m, n = len(M1), len(M1[0])
for top in range(m):
    col_sum = [0] * n                      # bắt đầu lại cho mỗi top
    for bottom in range(top, m):
        for c in range(n):
            col_sum[c] += M1[bottom][c]    # thêm đúng một hàng
        print(f"top = {top}, bottom = {bottom}: col_sum = {col_sum}")

top = 0, bottom = 0: col_sum = [1, 0, 1]
top = 0, bottom = 1: col_sum = [1, -2, 4]
top = 1, bottom = 1: col_sum = [0, -2, 3]


## 9. Bài con 1D: tổng đoạn con lớn nhất mà `<= k`

**Phát biểu:** cho mảng `arr` (có thể có số âm) và `k`. Trong mọi đoạn con **khác rỗng** `arr[l..r]` có tổng `<= k`, trả về tổng lớn nhất.

Ví dụ xuyên suốt: `arr = [5, -6, 3, -2, 4]`, `k = 3`.

Bảng mọi đoạn con (hàng = `l`, cột = `r`):

| `l \ r` | 0 | 1 | 2 | 3 | 4 |
|---|---:|---:|---:|---:|---:|
| **0** | 5 | -1 | 2 | 0 | 4 |
| **1** | | -6 | -3 | -5 | -1 |
| **2** | | | **3** | 1 | 5 |
| **3** | | | | -2 | 2 |
| **4** | | | | | 4 |

Các tổng `<= 3`: `-1, 2, 0, -6, -3, -5, -1, 3, 1, -2, 2` ⇒ lớn nhất là **`3`** (đoạn `arr[2..2] = [3]`). Để ý tổng lớn nhất không ràng buộc là `5` (`[5]` hoặc `[3, -2, 4]`) — vượt `k`.

### Baseline 1D: thử mọi `(l, r)` với prefix sum — `O(n²)`

Dùng `P` của mục 4: `sum(arr[l..r]) = P[r+1] - P[l]`. Hai vòng lặp ⇒ `O(n²)`. Ghép vào bài 2D thì vẫn là `O(m²·n²)` như Cách 1 — chưa tiến bộ gì, nhưng là bản tham chiếu để test.

In [11]:
def max_subarray_at_most_k_brute(arr, k):
    P = prefix_1d(arr)
    best = float("-inf")
    for l in range(len(arr)):
        for r in range(l, len(arr)):
            s = P[r + 1] - P[l]
            if s <= k and s > best:
                best = s
    return best


print(max_subarray_at_most_k_brute([5, -6, 3, -2, 4], 3))   # 3
print(max_subarray_at_most_k_brute([1, -2, 4], 2))          # 2  (top=0, bottom=1 của Ví dụ 1)

3
2


## 10. Insight 1D: đổi bài "tìm đoạn" thành bài "tìm một prefix"

Duyệt `r` từ trái sang phải. Đặt `cur = P[r+1]` (tổng từ đầu tới `r`). Mọi đoạn **kết thúc tại `r`** có dạng `arr[l..r]` với tổng `cur - P[l]`, trong đó `l` chạy từ `0` tới `r`, tức `P[l]` là **một prefix đã gặp trước đó** (`P[0], ..., P[r]`).

Điều kiện và mục tiêu:

```text
cur - P[l] <= k        <=>     P[l] >= cur - k
muốn  cur - P[l]  LỚN NHẤT   <=>   muốn  P[l]  NHỎ NHẤT
```

Gộp lại: **trong các prefix đã gặp, tìm prefix nhỏ nhất mà vẫn `>= cur - k`.** Giá trị đó thường gọi là **ceiling** (trần) của `cur - k`.

- Nếu tìm được prefix `p` như vậy ⇒ đoạn tốt nhất kết thúc tại `r` có tổng `cur - p` (và chắc chắn `<= k`).
- Nếu không có prefix nào `>= cur - k` ⇒ **không có** đoạn nào kết thúc tại `r` có tổng `<= k`.

**Ví dụ số:** `arr = [5, -6, 3, -2, 4]`, `k = 3`, xét `r = 2`:
`cur = P[3] = 2`, các prefix đã gặp: `P[0..2] = {0, 5, -1}`, `cur - k = -1`.
Prefix nhỏ nhất `>= -1` là `-1` (chính là `P[2]`) ⇒ tổng `2 - (-1) = 3` = đoạn `arr[2..2] = [3]`. ✔

Ô code dưới làm đúng việc đó bằng cách **quét tuyến tính** các prefix đã gặp — chưa nhanh, chỉ để kiểm chứng ý tưởng.

In [12]:
def best_ending_here_linear(seen_prefixes, cur, k):
    """Prefix nhỏ nhất >= cur - k (quét hết, O(số prefix)). Trả về (prefix, tổng đoạn) hoặc None."""
    candidates = [p for p in seen_prefixes if p >= cur - k]
    if not candidates:
        return None
    p = min(candidates)
    return p, cur - p


print(best_ending_here_linear([0, 5, -1], cur=2, k=3))        # (-1, 3)
print(best_ending_here_linear([0], cur=5, k=3))               # None: [5] > 3
print(best_ending_here_linear([0, 5, -1, 2, 0], cur=4, k=3))  # (2, 2): đoạn arr[3..4] = [-2, 4]

(-1, 3)
None
(2, 2)


Quét tuyến tính tốn `O(r)` mỗi bước ⇒ tổng `O(n²)` — bằng baseline. Nhưng câu hỏi "**số nhỏ nhất `>= x`** trong một tập" là câu hỏi kinh điển mà **binary search** trên một **danh sách đã sắp xếp** trả lời trong `O(log n)`. Ta học công cụ đó ngay sau đây.

## 11. Nền tảng 4: binary search và module `bisect`

### 11.1. Binary search (tìm kiếm nhị phân) là gì?

Trò chơi "đoán số từ 1 tới 100, được gợi ý lớn hơn / nhỏ hơn": đoán 50 → "lớn hơn" → đoán 75 → ... Mỗi lần **loại một nửa** ⇒ chỉ cần khoảng `log₂(100) ≈ 7` lần. Điều kiện để làm được: dữ liệu phải **đã sắp xếp**.

### 11.2. `bisect_left(a, x)`

Module chuẩn `bisect` của Python làm binary search trên `list` **đã sắp xếp tăng dần**:

- `bisect.bisect_left(a, x)` trả về **vị trí đầu tiên `i` mà `a[i] >= x`** (nếu mọi phần tử đều `< x` thì trả về `len(a)`).
- Nói cách khác: đó là vị trí sẽ chèn `x` vào để `a` vẫn sắp xếp, và nếu `x` đã có thì chèn **bên trái** các bản sao.
- `bisect.bisect_right(a, x)` tương tự nhưng là vị trí đầu tiên `a[i] > x` (chèn **bên phải** các bản sao).

Hệ quả cực kỳ quan trọng cho bài này: **`a[bisect_left(a, x)]` chính là "số nhỏ nhất `>= x`"** (ceiling của `x`) — miễn là vị trí đó `< len(a)`.

Ví dụ với `a = [-1, 0, 0, 2, 5]`:

| `x` | `bisect_left` | `a[i]` (ceiling) | `bisect_right` |
|---:|---:|---:|---:|
| 1 | 3 | 2 | 3 |
| 0 | 1 | 0 | 3 |
| -4 | 0 | -1 | 0 |
| 6 | 5 | (không có — vượt cuối) | 5 |

In [13]:
a = [-1, 0, 0, 2, 5]
for x in [1, 0, -4, 6]:
    i = bisect.bisect_left(a, x)
    ceil = a[i] if i < len(a) else "không có"
    print(f"x = {x:>2}: bisect_left = {i}, ceiling = {ceil}, bisect_right = {bisect.bisect_right(a, x)}")

x =  1: bisect_left = 3, ceiling = 2, bisect_right = 3
x =  0: bisect_left = 1, ceiling = 0, bisect_right = 3
x = -4: bisect_left = 0, ceiling = -1, bisect_right = 0
x =  6: bisect_left = 5, ceiling = không có, bisect_right = 5


### 11.3. Tự cài `bisect_left` để hiểu bên trong

Giữ một khoảng `[lo, hi)` (nửa mở: gồm `lo`, không gồm `hi`) mà đáp án chắc chắn nằm trong. Mỗi bước xem phần tử giữa `mid`:

- Nếu `a[mid] < x` ⇒ `mid` và mọi thứ bên trái đều `< x` ⇒ đáp án ở bên phải: `lo = mid + 1`.
- Ngược lại `a[mid] >= x` ⇒ `mid` có thể là đáp án, nhưng có thể còn vị trí sớm hơn ⇒ `hi = mid`.

Dừng khi `lo == hi`. **Invariant:** mọi chỉ số `< lo` có giá trị `< x`; mọi chỉ số `>= hi` có giá trị `>= x`. Mỗi bước khoảng giảm một nửa ⇒ `O(log n)`.

In [14]:
def my_bisect_left(a, x):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo


random.seed(1)
for _ in range(1000):
    arr = sorted(random.randint(-5, 5) for _ in range(random.randint(0, 8)))
    x = random.randint(-7, 7)
    assert my_bisect_left(arr, x) == bisect.bisect_left(arr, x)
print("my_bisect_left khớp bisect.bisect_left trên 1000 test ✔")
print(my_bisect_left([-1, 0, 0, 2, 5], 1))   # 3

my_bisect_left khớp bisect.bisect_left trên 1000 test ✔
3


### 11.4. Chèn mà vẫn giữ thứ tự: `bisect.insort` — và cái giá phải trả

`bisect.insort(a, x)` = tìm vị trí bằng `bisect_right` (`O(log n)`) rồi `a.insert(i, x)`.

Nhưng `list` của Python là **mảng liền kề trong bộ nhớ**: chèn vào giữa phải **dời** mọi phần tử phía sau sang phải một ô ⇒ bước chèn tốn **`O(n)`** trong trường hợp xấu (dù việc dời là thao tác chép bộ nhớ rất nhanh, viết bằng C).

Tóm lại với list sắp xếp:

| Thao tác | Chi phí |
|---|---|
| Tìm ceiling bằng `bisect_left` | `O(log n)` |
| Chèn bằng `insort` | `O(log n)` tìm + **`O(n)`** dời phần tử |

Muốn chèn thật sự `O(log n)` cần **cây tìm kiếm cân bằng** (balanced BST, ví dụ `TreeSet` của Java, `std::set` của C++) hoặc thư viện ngoài `sortedcontainers.SortedList` (không phải thư viện chuẩn, nên notebook này không dùng). Trên LeetCode với `n <= 100`, list + `insort` là đủ nhanh.

In [15]:
sorted_prefix = [0]
for v in [5, -1, 2, 0, 4]:
    bisect.insort(sorted_prefix, v)
    print(f"insort {v:>2} -> {sorted_prefix}")

insort  5 -> [0, 5]
insort -1 -> [-1, 0, 5]
insort  2 -> [-1, 0, 2, 5]
insort  0 -> [-1, 0, 0, 2, 5]
insort  4 -> [-1, 0, 0, 2, 4, 5]


## 12. Thuật toán 1D hoàn chỉnh: prefix sum + list sắp xếp + `bisect_left`

Ghép insight mục 10 với công cụ mục 11:

```python
sorted_prefix = [0]        # P[0] = 0 đã "gặp" sẵn -> cho phép đoạn bắt đầu từ chỉ số 0
cur = 0
best = -inf
for x in arr:
    cur += x                                   # cur = P[r+1]
    i = bisect_left(sorted_prefix, cur - k)    # prefix nhỏ nhất >= cur - k
    if i < len(sorted_prefix):
        best = max(best, cur - sorted_prefix[i])
    insort(sorted_prefix, cur)                 # SAU khi hỏi mới thêm cur
return best
```

Ba chi tiết phải hiểu, không học thuộc:

1. **Khởi tạo `[0]`**: đó là `P[0]`. Thiếu nó thì không bao giờ xét được đoạn bắt đầu từ chỉ số `0` (ví dụ `arr = [2, -1], k = 2` sẽ trả sai — xem phần Lỗi).
2. **Hỏi trước, thêm sau**: nếu thêm `cur` vào trước rồi mới hỏi, có thể tìm thấy chính `cur` ⇒ tổng `cur - cur = 0` = **đoạn rỗng**. Đề cấm hình rỗng; với `[[-3]], k = 0` sẽ trả `0` thay vì `-3`.
3. **`i < len(...)`**: nếu mọi prefix đều `< cur - k` thì `bisect_left` trả `len(...)` ⇒ không có đoạn hợp lệ kết thúc ở đây; phải kiểm tra trước khi đọc để tránh `IndexError`.

**Độ phức tạp (mảng độ dài `n`):** `n` lần `bisect_left` ⇒ `O(n log n)` cho phần tìm kiếm; `n` lần `insort` ⇒ `O(n log n)` so sánh + tới `O(n²)` lần dời phần tử trong trường hợp xấu nhất (chép bộ nhớ, rất nhanh). Với cây cân bằng thì cả bài 1D là đúng `O(n log n)`. Bộ nhớ `O(n)`.

In [16]:
def max_subarray_at_most_k(arr, k):
    sorted_prefix = [0]
    cur = 0
    best = float("-inf")
    for x in arr:
        cur += x
        i = bisect.bisect_left(sorted_prefix, cur - k)
        if i < len(sorted_prefix):
            best = max(best, cur - sorted_prefix[i])
        bisect.insort(sorted_prefix, cur)
    return best


print(max_subarray_at_most_k([5, -6, 3, -2, 4], 3))   # 3
print(max_subarray_at_most_k([1, -2, 4], 2))          # 2
print(max_subarray_at_most_k([2, 2, -1], 3))          # 3  (Ví dụ 2 chính là bài 1D!)
print(max_subarray_at_most_k([-3], 0))                # -3

3
2
3
-3


## 13. Trace kỹ bài 1D: `arr = [5, -6, 3, -2, 4]`, `k = 3`

Mảng prefix: `P = [0, 5, -1, 2, 0, 4]`. Ý nghĩa các cột:

- `r`, `x`: chỉ số và giá trị đang xét; `cur = P[r+1]`.
- `need = cur - k`: ta cần prefix **`>= need`**.
- `sorted_prefix` **trước** khi thêm `cur` (chứa đúng `P[0..r]`, đã sắp xếp).
- `i = bisect_left(...)`, `ceiling = sorted_prefix[i]` nếu `i` hợp lệ; tổng ứng viên `cur - ceiling`; đoạn tương ứng.

| `r` | `x` | `cur` | `need` | `sorted_prefix` trước | `i` | ceiling | `cur - ceiling` | đoạn | `best` |
|---:|---:|---:|---:|---|---:|---:|---:|---|---:|
| 0 | 5 | 5 | 2 | `[0]` | 1 | — (vượt cuối) | — | không có | `-inf` |
| 1 | -6 | -1 | -4 | `[0, 5]` | 0 | 0 | -1 | `arr[0..1]` | -1 |
| 2 | 3 | 2 | -1 | `[-1, 0, 5]` | 0 | -1 | **3** | `arr[2..2]` | **3** |
| 3 | -2 | 0 | -3 | `[-1, 0, 2, 5]` | 0 | -1 | 1 | `arr[2..3]` | 3 |
| 4 | 4 | 4 | 1 | `[-1, 0, 0, 2, 5]` | 3 | 2 | 2 | `arr[3..4]` | 3 |

Kết quả: **`3`**, khớp bảng mọi đoạn con ở mục 9.

Những chỗ đáng chú ý:

- **`r = 0`**: cần prefix `>= 2` nhưng chỉ có `0` ⇒ không có đoạn nào kết thúc tại 0 mà `<= 3` (đúng: `[5] = 5 > 3`). Đây là nhánh `i == len(...)`.
- **`r = 1`**: `best` trở thành **số âm** `-1` — nhờ khởi tạo `-inf` chứ không phải `0`.
- **`r = 2`**: ceiling `-1` là `P[2]` ⇒ `l = 2`, đoạn chỉ có `[3]`. Đây là lúc đạt đáp án.
- **`r = 4`**: `need = 1`, các prefix `-1, 0, 0` bị loại vì `< 1` (dùng chúng sẽ cho tổng `5, 4, 4 > 3`); ceiling là `2 = P[3]` ⇒ đoạn `[-2, 4]` tổng `2`. Prefix `5` cũng `>= 1` nhưng **lớn hơn** `2` nên cho tổng nhỏ hơn (`4 - 5 = -1`) — vì thế ta lấy prefix **nhỏ nhất** `>= need`.
- Prefix `0` xuất hiện **hai lần** (`P[0]` và `P[4]`) — list giữ cả hai bản, không sao cả.

Ô code dưới in đúng bảng trên (thêm cột `l` bằng cách tra lại vị trí của ceiling trong `P`).

In [17]:
def trace_subarray_at_most_k(arr, k):
    sorted_prefix = [0]
    P = [0]                        # chỉ để in ra l (vị trí bắt đầu đoạn)
    cur = 0
    best = float("-inf")
    for r, x in enumerate(arr):
        cur += x
        need = cur - k
        before = list(sorted_prefix)
        i = bisect.bisect_left(sorted_prefix, need)
        if i < len(sorted_prefix):
            ceil = sorted_prefix[i]
            l = P.index(ceil)
            best = max(best, cur - ceil)
            info = f"ceiling = {ceil:>2}, tổng = {cur - ceil:>2}, đoạn arr[{l}..{r}]"
        else:
            info = "không có prefix >= need -> không có đoạn hợp lệ"
        print(f"r={r} x={x:>2} cur={cur:>2} need={need:>2} sorted={before} i={i} | {info} | best={best}")
        bisect.insort(sorted_prefix, cur)
        P.append(cur)
    return best


print("Kết quả:", trace_subarray_at_most_k([5, -6, 3, -2, 4], 3))

r=0 x= 5 cur= 5 need= 2 sorted=[0] i=1 | không có prefix >= need -> không có đoạn hợp lệ | best=-inf
r=1 x=-6 cur=-1 need=-4 sorted=[0, 5] i=0 | ceiling =  0, tổng = -1, đoạn arr[0..1] | best=-1
r=2 x= 3 cur= 2 need=-1 sorted=[-1, 0, 5] i=0 | ceiling = -1, tổng =  3, đoạn arr[2..2] | best=3
r=3 x=-2 cur= 0 need=-3 sorted=[-1, 0, 2, 5] i=0 | ceiling = -1, tổng =  1, đoạn arr[2..3] | best=3
r=4 x= 4 cur= 4 need= 1 sorted=[-1, 0, 0, 2, 5] i=3 | ceiling =  2, tổng =  2, đoạn arr[3..4] | best=3
Kết quả: 3


### Test chéo bài 1D với baseline

Trước khi ghép vào bài 2D, ta phải chắc chắn bài 1D đúng — kể cả với số âm, `k` âm, mảng 1 phần tử, các prefix trùng nhau. Sinh mảng ngẫu nhiên nhỏ (seed cố định để chạy lại ra y hệt) và so với baseline `O(n²)`. Chỉ test những `k` mà có ít nhất một đoạn `<= k` (giống bảo đảm của đề).

In [18]:
random.seed(2024)
tested = 0
for _ in range(3000):
    arr = [random.randint(-6, 6) for _ in range(random.randint(1, 8))]
    k = random.randint(-8, 10)
    if min(arr) > k and all(sum(arr[l:r + 1]) > k for l in range(len(arr)) for r in range(l, len(arr))):
        continue                     # không có đoạn nào <= k: đề không cho phép, bỏ qua
    assert max_subarray_at_most_k(arr, k) == max_subarray_at_most_k_brute(arr, k), (arr, k)
    tested += 1
print(f"max_subarray_at_most_k khớp baseline trên {tested} mảng ngẫu nhiên ✔")

max_subarray_at_most_k khớp baseline trên 2430 mảng ngẫu nhiên ✔


## 14. Cách 2 (cách chính) — Cố định cặp hàng + nén cột + bài 1D

Ghép mục 8 (nén cột) với mục 12 (bài 1D):

```text
best = -inf
với mỗi top = 0 .. m-1:
    col_sum = [0] * n
    với mỗi bottom = top .. m-1:
        col_sum[c] += matrix[bottom][c]  với mọi c        # O(n)
        best = max(best, max_subarray_at_most_k(col_sum, k))   # bài 1D
return best
```

Mỗi hình chữ nhật `(top, bottom, left, right)` được xét **đúng một lần**: ở vòng `(top, bottom)` của nó, dưới dạng đoạn `col_sum[left..right]` của bài 1D. Vậy lấy max qua mọi cặp hàng là đúng đáp án.

In [19]:
def max_sum_submatrix_rows_bisect(matrix, k):
    m, n = len(matrix), len(matrix[0])
    best = float("-inf")
    for top in range(m):
        col_sum = [0] * n
        for bottom in range(top, m):
            row = matrix[bottom]
            for c in range(n):
                col_sum[c] += row[c]
            best = max(best, max_subarray_at_most_k(col_sum, k))
    return best


print(max_sum_submatrix_rows_bisect([[1, 0, 1], [0, -2, 3]], 2))   # 2
print(max_sum_submatrix_rows_bisect([[2, 2, -1]], 3))              # 3
print(max_sum_submatrix_rows_bisect([[-5, -2]], -1))               # -2

2
3
-2


### Trace Cách 2 trên Ví dụ 1 (`k = 2`)

| `top` | `bottom` | `col_sum` | Kết quả bài 1D (đoạn tốt nhất) | Hình chữ nhật | `best` |
|---:|---:|---|---|---|---:|
| 0 | 0 | `[1, 0, 1]` | 2 (`cột 0..2`) | hàng 0..0, cột 0..2 | 2 |
| 0 | 1 | `[1, -2, 4]` | 2 (`cột 1..2`: `-2 + 4`) | hàng 0..1, cột 1..2 — **hình xanh của đề** | 2 |
| 1 | 1 | `[0, -2, 3]` | 1 (`cột 0..2` hoặc `cột 1..2`) | hàng 1..1 | 2 |

Ở cặp `(0, 1)`, tổng cả dải `1 - 2 + 4 = 3 > 2` bị loại; bài 1D tìm ra đoạn `[-2, 4]`. Kết quả cuối `2`. Ô code dưới in cùng nội dung (dùng bản trace của bài 1D để lấy được vị trí cột).

In [20]:
def best_segment_at_most_k(arr, k):
    """Như max_subarray_at_most_k nhưng trả thêm (l, r) của đoạn tốt nhất — chỉ dùng để in trace."""
    sorted_prefix, P, cur = [0], [0], 0
    best, where = float("-inf"), None
    for r, x in enumerate(arr):
        cur += x
        i = bisect.bisect_left(sorted_prefix, cur - k)
        if i < len(sorted_prefix) and cur - sorted_prefix[i] > best:
            best, where = cur - sorted_prefix[i], (P.index(sorted_prefix[i]), r)
        bisect.insort(sorted_prefix, cur)
        P.append(cur)
    return best, where


def trace_max_sum_submatrix(matrix, k):
    m, n = len(matrix), len(matrix[0])
    best = float("-inf")
    for top in range(m):
        col_sum = [0] * n
        for bottom in range(top, m):
            for c in range(n):
                col_sum[c] += matrix[bottom][c]
            s, where = best_segment_at_most_k(col_sum, k)
            best = max(best, s)
            seg = f"cột {where[0]}..{where[1]}" if where else "không có"
            print(f"top={top} bottom={bottom} col_sum={col_sum} -> 1D = {s} ({seg}) | best = {best}")
    return best


print("Kết quả:", trace_max_sum_submatrix([[1, 0, 1], [0, -2, 3]], 2))

top=0 bottom=0 col_sum=[1, 0, 1] -> 1D = 2 (cột 0..2) | best = 2
top=0 bottom=1 col_sum=[1, -2, 4] -> 1D = 2 (cột 1..2) | best = 2
top=1 bottom=1 col_sum=[0, -2, 3] -> 1D = 1 (cột 0..2) | best = 2
Kết quả: 2


### Độ phức tạp Cách 2 — nói cho thật chính xác

Gọi `m` = số hàng, `n` = số cột.

- Số cặp hàng: `m(m+1)/2 = O(m²)`.
- Mỗi cặp: cập nhật `col_sum` `O(n)` + bài 1D: `n` lần `bisect_left` `O(log n)` và `n` lần `insort`.
- **Phần tìm kiếm**: `O(m²·n log n)`. Đây là độ phức tạp "chuẩn" của thuật toán khi dùng một cấu trúc chèn `O(log n)` (cây cân bằng / `TreeSet`).
- **Với `list` + `insort`** của Python: mỗi lần chèn có thể dời tới `O(n)` phần tử ⇒ trên lý thuyết, trường hợp xấu nhất là `O(m²·n²)`. Tuy nhiên việc dời là một lệnh chép bộ nhớ (memmove) cực nhanh trong C, còn `n <= 100`, nên trên thực tế phần tốn thời gian là các phép `bisect` và vòng lặp Python — chạy nhanh hơn Cách 1 rất nhiều.
- **Bộ nhớ**: `O(n)` cho `col_sum` và `sorted_prefix` (không cần bảng prefix 2D).

So với Cách 1 (`O(m²·n²)` **vòng lặp Python**), Cách 2 thay vòng `O(n²)` theo cặp cột bằng `O(n log n)` tìm kiếm.

## 15. Nền tảng 5: thuật toán Kadane (tổng đoạn con lớn nhất, không ràng buộc)

Bài LeetCode 53 (Maximum Subarray): tìm tổng lớn nhất của một đoạn con khác rỗng — **không có trần `k`**. Kadane giải trong `O(n)` với một ý rất đời thường:

> Gọi `ending` = tổng lớn nhất của đoạn **kết thúc tại vị trí hiện tại**. Khi gặp `x`, có hai lựa chọn: **nối** `x` vào đoạn tốt nhất trước đó (`ending + x`) hoặc **bắt đầu lại** từ `x`. Chọn cái lớn hơn: `ending = max(x, ending + x)`.

Nếu `ending` trước đó âm thì nối vào chỉ làm thiệt ⇒ bắt đầu lại. Đáp án là max của mọi `ending`.

**Ví dụ:** `arr = [5, -6, 3, -2, 4]`

| `x` | `ending = max(x, ending + x)` | `best` |
|---:|---|---:|
| 5 | `5` | 5 |
| -6 | `max(-6, -1) = -1` | 5 |
| 3 | `max(3, 2) = 3` (bắt đầu lại) | 5 |
| -2 | `max(-2, 1) = 1` | 5 |
| 4 | `max(4, 5) = 5` | 5 |

Kadane cho `5`, nhưng `5 > k = 3`, nên với bài của ta Kadane **một mình** là chưa đủ.

In [21]:
def kadane(arr):
    ending = best = arr[0]
    for x in arr[1:]:
        ending = max(x, ending + x)
        best = max(best, ending)
    return best


print(kadane([5, -6, 3, -2, 4]))   # 5
print(kadane([1, 0, 1]))           # 2
print(kadane([-4, -2, -7]))        # -2  (mảng toàn âm: chọn một phần tử lớn nhất)

5
2
-2


## 16. Cách 3 — Ba tối ưu thực tế

### Tối ưu 1: dùng Kadane làm "bộ lọc nhanh"

Với một cặp hàng, gọi `K = kadane(col_sum)` = tổng lớn nhất của **mọi** đoạn con.

- Nếu `K <= k`: đoạn tốt nhất **không ràng buộc** đã thỏa trần ⇒ nó cũng chính là đoạn tốt nhất **trong các đoạn `<= k`** (vì không đoạn nào lớn hơn `K`). Lấy luôn `K`, **bỏ qua** bước `bisect` `O(n log n)` — chỉ tốn `O(n)`.
- Nếu `K > k`: Kadane không giúp gì, chạy bài 1D đầy đủ như Cách 2.

Khi `k` lớn (so với các tổng), rất nhiều cặp hàng rơi vào trường hợp đầu ⇒ nhanh hơn hẳn.

### Tối ưu 2: dừng sớm khi `best == k`

Đáp án không bao giờ vượt `k`. Hễ tìm được tổng **đúng bằng `k`** thì không thể tốt hơn ⇒ `return k` ngay. (Ví dụ 1 sẽ dừng ngay ở cặp hàng đầu tiên.)

### Tối ưu 3: cố định chiều **nhỏ hơn** — trả lời Follow-up

Cách 2 tốn `O(m²·n log n)`: chiều được **cố định cặp** bị **bình phương**, chiều còn lại chỉ chịu `n log n`. Nếu số hàng rất lớn so với số cột (Follow-up của đề, ví dụ `m = 10000, n = 10`), ta nên **cố định cặp cột** và nén **hàng**: `O(n²·m log m)` — nhỏ hơn rất nhiều.

Cách cài gọn nhất: nếu `m > n` thì **chuyển vị** (transpose) ma trận — đổi hàng thành cột — rồi chạy y nguyên thuật toán. Tổng của mọi hình chữ nhật không đổi khi chuyển vị. Tổng quát: `O(min(m,n)² · max(m,n) · log max(m,n))`.

### Công cụ: chuyển vị ma trận bằng `zip(*matrix)`

- `*matrix` "rải" các hàng ra làm các đối số riêng: `zip(*[[1,2,3],[4,5,6]])` giống `zip([1,2,3], [4,5,6])`.
- `zip` ghép phần tử thứ `i` của mọi hàng lại thành một bộ (`tuple`) ⇒ ra `(1,4), (2,5), (3,6)` — chính là các **cột**.
- Bọc `list(...)` để mỗi cột thành `list`.

In [22]:
mat = [[1, 2, 3],
       [4, 5, 6]]
t = [list(col) for col in zip(*mat)]
print("ban đầu  (2 x 3):", mat)
print("chuyển vị (3 x 2):", t)

# Tổng mọi hình chữ nhật không đổi khi chuyển vị -> đáp án không đổi
M1 = [[1, 0, 1], [0, -2, 3]]
M1_t = [list(col) for col in zip(*M1)]
print(max_sum_naive(M1, 2), max_sum_naive(M1_t, 2))   # 2 2

ban đầu  (2 x 3): [[1, 2, 3], [4, 5, 6]]
chuyển vị (3 x 2): [[1, 4], [2, 5], [3, 6]]
2 2


In [23]:
def max_sum_submatrix_fast(matrix, k):
    m, n = len(matrix), len(matrix[0])
    if m > n:                                   # Tối ưu 3: cặp lặp chạy trên chiều nhỏ hơn
        matrix = [list(col) for col in zip(*matrix)]
        m, n = n, m
    best = float("-inf")
    for top in range(m):
        col_sum = [0] * n
        for bottom in range(top, m):
            row = matrix[bottom]
            for c in range(n):
                col_sum[c] += row[c]
            # Tối ưu 1: Kadane — nếu tổng lớn nhất không ràng buộc đã <= k thì lấy luôn
            K = kadane(col_sum)
            if K <= k:
                best = max(best, K)
            else:
                best = max(best, max_subarray_at_most_k(col_sum, k))
            if best == k:                       # Tối ưu 2: không thể tốt hơn k
                return k
    return best


print(max_sum_submatrix_fast([[1, 0, 1], [0, -2, 3]], 2))   # 2
print(max_sum_submatrix_fast([[2, 2, -1]], 3))              # 3
print(max_sum_submatrix_fast([[2], [-1], [3]], 3))          # 3 (m = 3 > n = 1 -> chuyển vị)

2
3
3


### Kadane bỏ qua được bao nhiêu lần?

Thí nghiệm nhỏ: trên ma trận ngẫu nhiên `20 x 20` giá trị trong `[-100, 100]`, đếm xem bao nhiêu cặp hàng được Kadane "cứu" (không phải chạy `bisect`), với vài giá trị `k` khác nhau. (Ở đây tắt dừng sớm để đếm đủ mọi cặp.) Bạn sẽ thấy: `k` càng lớn thì Kadane càng hay đủ dùng; `k` nhỏ/âm thì gần như luôn phải chạy bài 1D đầy đủ — đó là lý do Kadane chỉ là **tối ưu thực tế**, không đổi độ phức tạp trường hợp xấu.

In [24]:
def count_kadane_skips(matrix, k):
    m, n = len(matrix), len(matrix[0])
    skipped = total = 0
    for top in range(m):
        col_sum = [0] * n
        for bottom in range(top, m):
            for c in range(n):
                col_sum[c] += matrix[bottom][c]
            total += 1
            if kadane(col_sum) <= k:
                skipped += 1
    return skipped, total


random.seed(7)
big = [[random.randint(-100, 100) for _ in range(20)] for _ in range(20)]
for k in [-100, 0, 500, 2000, 5000]:
    s, t = count_kadane_skips(big, k)
    print(f"k = {k:>5}: Kadane bỏ qua {s:>3}/{t} cặp hàng")

k =  -100: Kadane bỏ qua   0/210 cặp hàng
k =     0: Kadane bỏ qua   0/210 cặp hàng
k =   500: Kadane bỏ qua 110/210 cặp hàng
k =  2000: Kadane bỏ qua 210/210 cặp hàng
k =  5000: Kadane bỏ qua 210/210 cặp hàng


## 17. Code nộp LeetCode

Bản nộp gộp Cách 2 + ba tối ưu của Cách 3, viết gọn trong `class Solution` với đúng signature `maxSumSubmatrix(self, matrix, k)`. Hàm Kadane và bài 1D viết thành **hàm lồng** (nested function) bên trong method để bản nộp tự đủ. `from typing import List` và `bisect` LeetCode đã import sẵn; khi chạy ở máy mình thì cần import (đã làm ở đầu notebook).

In [25]:
class Solution:
    def maxSumSubmatrix(self, matrix: List[List[int]], k: int) -> int:
        m, n = len(matrix), len(matrix[0])
        if m > n:                                   # cặp lặp trên chiều nhỏ hơn
            matrix = [list(col) for col in zip(*matrix)]
            m, n = n, m

        def best_at_most_k(arr):
            # Kadane: nếu tổng đoạn lớn nhất đã <= k thì đó là đáp án của mảng này
            ending = mx = arr[0]
            for x in arr[1:]:
                ending = max(x, ending + x)
                mx = max(mx, ending)
            if mx <= k:
                return mx
            # prefix sum + list sắp xếp: tìm prefix nhỏ nhất >= cur - k
            sorted_prefix = [0]
            cur, res = 0, float("-inf")
            for x in arr:
                cur += x
                i = bisect.bisect_left(sorted_prefix, cur - k)
                if i < len(sorted_prefix):
                    res = max(res, cur - sorted_prefix[i])
                bisect.insort(sorted_prefix, cur)
            return res

        best = float("-inf")
        for top in range(m):
            col_sum = [0] * n
            for bottom in range(top, m):
                row = matrix[bottom]
                for c in range(n):
                    col_sum[c] += row[c]
                best = max(best, best_at_most_k(col_sum))
                if best == k:
                    return k
        return best

In [26]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<38} PASS ({actual})")


sol = Solution()
check("Ví dụ 1", sol.maxSumSubmatrix([[1, 0, 1], [0, -2, 3]], 2), 2)
check("Ví dụ 2", sol.maxSumSubmatrix([[2, 2, -1]], 3), 3)
check("một ô, bằng k", sol.maxSumSubmatrix([[5]], 5), 5)
check("một ô âm, k = 0 (không trả 0!)", sol.maxSumSubmatrix([[-3]], 0), -3)
check("k âm", sol.maxSumSubmatrix([[-5, -2]], -1), -2)
check("một cột (bị chuyển vị)", sol.maxSumSubmatrix([[2], [-1], [3]], 3), 3)
check("toàn âm, k lớn", sol.maxSumSubmatrix([[-1, -2], [-3, -4]], 100), -1)
check("k nhỏ hơn mọi tổng dương", sol.maxSumSubmatrix([[2, 2], [2, 2]], 3), 2)
check("cần bỏ số lớn ở đầu", sol.maxSumSubmatrix([[4, -3, 2]], 3), 3)
check("giá trị biên ±100", sol.maxSumSubmatrix([[100, -100], [-100, 100]], 0), 0)

Ví dụ 1                                PASS (2)
Ví dụ 2                                PASS (3)
một ô, bằng k                          PASS (5)
một ô âm, k = 0 (không trả 0!)         PASS (-3)
k âm                                   PASS (-2)
một cột (bị chuyển vị)                 PASS (3)
toàn âm, k lớn                         PASS (-1)
k nhỏ hơn mọi tổng dương               PASS (2)
cần bỏ số lớn ở đầu                    PASS (3)
giá trị biên ±100                      PASS (0)


### Test chéo mọi cách trên input ngẫu nhiên

Đối chiếu 5 lời giải — `max_sum_naive` (Cách 0, trọng tài), `max_sum_prefix2d` (Cách 1), `max_sum_submatrix_rows_bisect` (Cách 2), `max_sum_submatrix_fast` (Cách 3) và `Solution` — trên nhiều ma trận nhỏ ngẫu nhiên, gồm cả ma trận "cao" (`m > n`) để kiểm tra nhánh chuyển vị, và `k` âm/dương. Chọn `k` sao cho luôn có ít nhất một hình `<= k` (đúng bảo đảm của đề): lấy `k` ngẫu nhiên nhưng không nhỏ hơn ô nhỏ nhất.

In [27]:
random.seed(363)
for _ in range(1500):
    m, n = random.randint(1, 5), random.randint(1, 5)
    mat = [[random.randint(-10, 10) for _ in range(n)] for _ in range(m)]
    lo = min(min(row) for row in mat)                 # ô nhỏ nhất -> hình 1 ô đó luôn <= k
    k = random.randint(lo, lo + 40)
    expected = max_sum_naive(mat, k)
    assert max_sum_prefix2d(mat, k) == expected
    assert max_sum_submatrix_rows_bisect(mat, k) == expected
    assert max_sum_submatrix_fast(mat, k) == expected
    assert Solution().maxSumSubmatrix(mat, k) == expected, (mat, k)
print("5 lời giải khớp nhau trên 1500 ma trận ngẫu nhiên ✔")

5 lời giải khớp nhau trên 1500 ma trận ngẫu nhiên ✔


### Thử ở kích thước lớn nhất của đề (`100 x 100`)

Đo thời gian thô (con số phụ thuộc máy, chỉ để cảm nhận) của Cách 2 và `Solution` trên ma trận `100 x 100` gần với **trường hợp xấu**:

- mọi ô là số **chẵn** trong `[-100, 100]` còn `k = -7` là số **lẻ** ⇒ không hình nào có tổng đúng bằng `k` ⇒ **không bao giờ dừng sớm**;
- `k` âm ⇒ tổng lớn nhất (Kadane) của hầu hết các cặp hàng là số dương `> k` ⇒ bộ lọc Kadane gần như **không giúp được**.

Vậy cả hai bản đều phải chạy bài 1D cho gần đủ `5050` cặp hàng. Cách 1 với `25,5 triệu` hình sẽ chậm hơn nhiều nên ta không chạy ở đây.

In [28]:
random.seed(100)
huge = [[2 * random.randint(-50, 50) for _ in range(100)] for _ in range(100)]   # toàn số chẵn
for name, fn in [("Cách 2", lambda: max_sum_submatrix_rows_bisect(huge, -7)),
                 ("Solution", lambda: Solution().maxSumSubmatrix(huge, -7))]:
    t0 = time.perf_counter()
    ans = fn()
    print(f"{name:<9} đáp án = {ans}, thời gian ≈ {time.perf_counter() - t0:.2f} giây")

Cách 2    đáp án = -8, thời gian ≈ 0.17 giây


Solution  đáp án = -8, thời gian ≈ 0.18 giây


## 18. Vì sao lời giải đúng?

### Tầng 2D: không bỏ sót, không nhầm

- Mỗi hình chữ nhật có **một** cặp hàng `(top, bottom)` duy nhất. Khi vòng lặp tới cặp đó, invariant của `col_sum` là: **`col_sum[c]` = tổng cột `c` từ hàng `top` tới hàng `bottom`** (đúng khi khởi tạo `[0]*n` rồi cộng dần từng hàng).
- Do đó tổng của hình `(top, bottom, left, right)` **bằng** tổng đoạn `col_sum[left..right]`. Tập các hình có dải hàng `(top, bottom)` ↔ tập các đoạn con của `col_sum` — tương ứng một-một.
- Bài 1D trả về max của các đoạn `<= k` ⇒ max qua mọi cặp hàng = max của mọi hình `<= k`.
- Chuyển vị không đổi tập tổng của các hình chữ nhật ⇒ không đổi đáp án.

### Tầng 1D: invariant của `sorted_prefix`

Ngay **trước** khi xét phần tử `arr[r]` (sau khi cộng `cur = P[r+1]`), `sorted_prefix` chứa **đúng** các giá trị `P[0], P[1], ..., P[r]` (đã sắp xếp) — mỗi giá trị ứng với một điểm bắt đầu `l` hợp lệ cho đoạn kết thúc tại `r`. (`P[0] = 0` có sẵn từ đầu; `P[r+1]` chỉ được thêm **sau** khi hỏi.)

Từ đó, ở bước `r`:

- Mọi đoạn khác rỗng kết thúc tại `r` là `cur - p` với `p` trong `sorted_prefix` — không thiếu, không thừa (không có đoạn rỗng vì `cur` chưa được thêm).
- `cur - p <= k ⇔ p >= cur - k`; trong các `p` đó, `cur - p` lớn nhất khi `p` nhỏ nhất = `sorted_prefix[bisect_left(sorted_prefix, cur - k)]`.
- Vậy mỗi bước ta tìm đúng **đoạn tốt nhất kết thúc tại `r`**; max qua mọi `r` là đáp án 1D.

### Các tối ưu không làm sai

- **Kadane**: nếu max mọi đoạn `K <= k` thì `K` cũng là max của các đoạn `<= k` (tập con chứa phần tử lớn nhất của tập lớn).
- **Dừng sớm**: mọi ứng viên đều `<= k`, nên khi `best == k` không gì vượt được.

## 19. Độ phức tạp và so sánh các cách

Gọi `m` = số hàng, `n` = số cột; với Cách 3 đặt `a = min(m, n)`, `b = max(m, n)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| 0 — Naive | thử mọi hình, cộng tay | `O(m³·n³)` | `O(1)` | chỉ để làm trọng tài |
| 1 — Prefix 2D | thử mọi hình, tổng `O(1)` | `O(m²·n²)` | `O(m·n)` | `≈ 25,5 triệu` hình khi `100 x 100` ⇒ TLE với Python |
| 2 — Nén cột + bisect | cố định cặp hàng, bài 1D bằng prefix + list sắp xếp | `O(m²·n log n)` tìm kiếm; `insort` trên list thêm tới `O(n)` mỗi lần chèn (lý thuyết `O(m²·n²)` trường hợp xấu, nhưng là memmove rất nhanh) | `O(n)` | cách chính |
| 3 — + Kadane, dừng sớm, chiều nhỏ | như Cách 2 + bỏ qua khi `kadane <= k` | `O(a²·b log b)` (cùng lưu ý `insort`); thường nhanh hơn nhiều | `O(b)` (+ `O(m·n)` nếu chuyển vị) | trả lời Follow-up |

Bài 1D riêng: baseline `O(n²)`; prefix + list sắp xếp `O(n log n)` tìm kiếm (`O(n log n)` hoàn toàn nếu dùng cây cân bằng); Kadane `O(n)` nhưng chỉ giải được khi không có trần `k`.

## 20. Lỗi thường gặp

### Lỗi 1 — Dùng cửa sổ trượt (two pointers) như bài mảng toàn số dương

Ý tưởng "mở rộng bên phải, nếu tổng `> k` thì co bên trái" chỉ đúng khi mọi số **không âm** (khi đó mở rộng chỉ làm tổng tăng). Ở bài này có số âm: một số lớn ở đầu tưởng như phải bỏ, nhưng sau đó một số âm lại kéo tổng xuống.

Phản ví dụ: `arr = [4, -3, 2]`, `k = 3`. Đáp án đúng là **`3`** (cả mảng `4 - 3 + 2`). Cửa sổ trượt gặp `4 > 3` liền bỏ `4` vĩnh viễn ⇒ không bao giờ xét lại đoạn chứa nó.

In [29]:
def wrong_sliding_window(arr, k):
    l, s, best = 0, 0, float("-inf")
    for r, x in enumerate(arr):
        s += x
        while s > k and l <= r:          # co trái khi vượt k — SAI khi có số âm
            s -= arr[l]
            l += 1
        if l <= r:
            best = max(best, s)
    return best


print("cửa sổ trượt:", wrong_sliding_window([4, -3, 2], 3), "| đúng:", max_subarray_at_most_k([4, -3, 2], 3))

cửa sổ trượt: -1 | đúng: 3


### Lỗi 2 — Quên `sorted_prefix = [0]` (thiếu `P[0]`)

Không có `P[0] = 0` thì không đoạn nào được phép **bắt đầu từ chỉ số 0**.

### Lỗi 3 — Thêm `cur` **trước** rồi mới hỏi

Khi đó `cur` có thể tự tìm thấy chính nó ⇒ tổng `cur - cur = 0` = đoạn **rỗng**. Nếu `k >= 0` mà mọi hình thật đều âm, ta trả `0` sai.

### Lỗi 4 — Khởi tạo `best = 0`

Đáp án có thể âm (`k` âm, hoặc mọi ô âm). Phải khởi tạo `float("-inf")`.

Ô code dưới đưa phản ví dụ cho từng lỗi.

In [30]:
def wrong_no_zero(arr, k):                   # Lỗi 2: thiếu P[0]
    sp, cur, best = [], 0, float("-inf")
    for x in arr:
        cur += x
        i = bisect.bisect_left(sp, cur - k)
        if i < len(sp):
            best = max(best, cur - sp[i])
        bisect.insort(sp, cur)
    return best


def wrong_insert_first(arr, k):              # Lỗi 3: thêm trước, hỏi sau
    sp, cur, best = [0], 0, float("-inf")
    for x in arr:
        cur += x
        bisect.insort(sp, cur)
        i = bisect.bisect_left(sp, cur - k)
        if i < len(sp):
            best = max(best, cur - sp[i])
    return best


def wrong_best_zero(arr, k):                 # Lỗi 4: best = 0
    sp, cur, best = [0], 0, 0
    for x in arr:
        cur += x
        i = bisect.bisect_left(sp, cur - k)
        if i < len(sp):
            best = max(best, cur - sp[i])
        bisect.insort(sp, cur)
    return best


print("Lỗi 2 [2, -1], k=2 :", wrong_no_zero([2, -1], 2), "| đúng:", max_subarray_at_most_k([2, -1], 2))
print("Lỗi 3 [-3],    k=0 :", wrong_insert_first([-3], 0), "| đúng:", max_subarray_at_most_k([-3], 0))
print("Lỗi 4 [-5, -2], k=-1:", wrong_best_zero([-5, -2], -1), "| đúng:", max_subarray_at_most_k([-5, -2], -1))

Lỗi 2 [2, -1], k=2 : -1 | đúng: 2
Lỗi 3 [-3],    k=0 : 0 | đúng: -3
Lỗi 4 [-5, -2], k=-1: 0 | đúng: -2


### Lỗi 5 — Tìm **sai chiều**: lấy prefix lớn nhất `<= cur - k`

Rất dễ nhầm điều kiện `cur - P[l] <= k` thành `P[l] <= cur - k`. Khi đó ta lại tìm **floor** (số lớn nhất `<= cur - k`) bằng `bisect_right(...) - 1` — và kết quả là những tổng **`>= k`**, ngược hẳn yêu cầu!

Cách tự kiểm tra: lấy một số cụ thể. `cur = 5, k = 3`: muốn `5 - p <= 3` ⇒ `p >= 2`. Chuyển vế phải đổi dấu: `-p <= k - cur` ⇒ `p >= cur - k`. Ô dưới cho thấy bản sai trả về giá trị **vượt `k`**.

### Lỗi 6 — Viết ngược bộ lọc Kadane

`if K > k: best = max(best, K)` là sai hoàn toàn (lấy một tổng vượt trần). Đúng là: `K <= k` ⇒ lấy `K`; ngược lại phải chạy bài 1D đầy đủ.

### Lỗi 7 — Chuyển vị nhưng quên đổi `m, n`

Sau `matrix = [list(col) for col in zip(*matrix)]` mà vẫn dùng `m, n` cũ ⇒ `IndexError` hoặc bỏ sót. Nhớ `m, n = n, m`.

In [31]:
def wrong_direction(arr, k):                 # Lỗi 5: tìm floor của cur - k
    sp, cur, best = [0], 0, float("-inf")
    for x in arr:
        cur += x
        i = bisect.bisect_right(sp, cur - k) - 1     # prefix lớn nhất <= cur - k
        if i >= 0:
            best = max(best, cur - sp[i])
        bisect.insort(sp, cur)
    return best


arr, k = [5, -6, 3, -2, 4], 3
print("bản sai chiều:", wrong_direction(arr, k), "(> k = 3 !) | đúng:", max_subarray_at_most_k(arr, k))

bản sai chiều: 5 (> k = 3 !) | đúng: 3


### Các lỗi "lặt vặt" khác

- Viết `for bottom in range(m)` thay vì `range(top, m)` ⇒ xét dải hàng ngược (`bottom < top`), và `col_sum` cộng sai.
- Quên tạo lại `col_sum = [0] * n` cho mỗi `top` mới ⇒ cộng dồn cả các hàng phía trên.
- Dùng `bisect_right` thay `bisect_left` trong bài 1D: vẫn đúng **giá trị** trong bài này? Không hẳn — `bisect_right(sp, cur - k)` bỏ qua các prefix **bằng đúng** `cur - k`, tức bỏ qua các đoạn có tổng **đúng bằng `k`** (đáp án tốt nhất có thể!). Phải dùng `bisect_left`.
- Tính `kadane` trên mảng rỗng (`arr[0]` lỗi) — đề bảo đảm `n >= 1` nên không xảy ra, nhưng nếu bạn tự viết biến thể thì nhớ kiểm tra.

In [32]:
def wrong_bisect_right(arr, k):
    sp, cur, best = [0], 0, float("-inf")
    for x in arr:
        cur += x
        i = bisect.bisect_right(sp, cur - k)          # SAI: bỏ qua prefix == cur - k
        if i < len(sp):
            best = max(best, cur - sp[i])
        bisect.insort(sp, cur)
    return best


print("bisect_right [2, 2, -1], k=3:", wrong_bisect_right([2, 2, -1], 3), "| đúng:", max_subarray_at_most_k([2, 2, -1], 3))

bisect_right [2, 2, -1], k=3: 2 | đúng: 3


## 21. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu → kỹ thuật

| Dấu hiệu trong đề | Nghĩ ngay tới |
|---|---|
| "tổng của một hình chữ nhật / ma trận con" | **prefix sum 2D** (bao hàm – loại trừ) |
| "tốt nhất trong mọi hình chữ nhật", `m, n` cỡ 100–200 | **cố định cặp hàng + nén cột** ⇒ bài 1D, `O(m²·...)` |
| "tổng đoạn con `<= k` / gần `k` nhất", có số **âm** | prefix sum + **cấu trúc sắp xếp** (`bisect`, `TreeSet`) tìm ceiling/floor |
| "tổng đoạn con **đúng bằng** `k`" | prefix sum + **hash map đếm** (`dict`) — không cần sắp xếp |
| "tổng đoạn con lớn nhất", không có trần | **Kadane** `O(n)` |
| mảng **toàn số dương**, "tổng `<= k`" | có thể dùng **cửa sổ trượt** — nhưng **không** khi có số âm |
| "số hàng >> số cột" | cố định cặp theo **chiều nhỏ** (chuyển vị) |

### Bài liên quan

- **#53 Maximum Subarray** — Kadane (bản 1D không trần).
- **#303 / #304 Range Sum Query (Immutable / 2D - Immutable)** — prefix sum 1D / 2D thuần túy (Bài tập 4).
- **#560 Subarray Sum Equals K** — prefix sum + hash map đếm.
- **#1074 Number of Submatrices That Sum to Target** — đúng khung "nén cột" của bài này + #560 (Bài tập 5).
- **#85 Maximal Rectangle** — cũng "cố định hàng đáy, nén thành 1D" (histogram).
- **#327 Count of Range Sum**, **#220 Contains Duplicate III** — prefix/giá trị + cấu trúc sắp xếp, tìm ceiling/floor.

## 22. Bài tập tự luyện

Làm theo thứ tự, mỗi bài khó hơn bài trước. Gợi ý chia 3 mức — chỉ mở mức tiếp theo khi bí. **Đáp án ở cuối notebook.**

### Bài 1 — Đoán output (không chạy code)

1. `bisect.bisect_left([-2, 0, 1, 4], 1)`, `bisect.bisect_left([-2, 0, 1, 4], 5)`, `bisect.bisect_left([-2, 0, 1, 4], -3)` lần lượt bằng bao nhiêu?
2. `prefix_1d([3, -1, 2])` trả về gì?
3. `build_prefix_2d([[1, 2], [3, 4]])` trả về gì?
4. `[list(c) for c in zip(*[[1, 2, 3], [4, 5, 6]])]` trả về gì?
5. Với `matrix = [[1, 2], [3, -4], [5, 6]]`, `col_sum` của cặp hàng `top = 1, bottom = 2` là gì?

Gợi ý mức 1: `bisect_left` = vị trí đầu tiên có giá trị `>= x`.
Gợi ý mức 2: `P` 1D dài `len + 1` và bắt đầu bằng `0`; `P` 2D có hàng 0 và cột 0 toàn `0`.
Gợi ý mức 3: câu 3 dùng `P[r+1][c+1] = trên + trái - góc + ô`; câu 5 cộng dọc hàng 1 và hàng 2.

### Bài 2 — Tính tay bài 1D

Lập bảng trace (giống mục 13) cho `arr = [2, -3, 4, -1, 3]`, `k = 4`. Cột: `r`, `x`, `cur`, `need`, `sorted_prefix` trước, `i`, ceiling, `cur - ceiling`, `best`. Đáp án là bao nhiêu và đạt ở đoạn nào? Kadane của mảng này bằng bao nhiêu — có dùng bộ lọc Kadane được không?

Gợi ý mức 1: `P = [0, 2, -1, 3, 2, 5]`.
Gợi ý mức 2: ở `r = 2`, `need = -1`, và `-1` đã có trong `sorted_prefix`.
Gợi ý mức 3: Kadane: đoạn `[4, -1, 3]` tổng `6`.

### Bài 3 — Điền TODO: bài 1D

Hoàn thiện hàm theo 4 TODO. Đừng nhìn lại mục 12!

Gợi ý mức 1: thứ tự mỗi bước: **cộng `cur` → hỏi → cập nhật `best` → thêm**.
Gợi ý mức 2: hỏi bằng `bisect.bisect_left(sorted_prefix, cur - k)`.
Gợi ý mức 3: nhớ kiểm tra `i < len(sorted_prefix)` trước khi đọc; thêm bằng `bisect.insort(sorted_prefix, cur)`.

In [33]:
def exercise_max_subarray_at_most_k(arr, k):
    sorted_prefix = [0]          # P[0] = 0
    cur = 0
    best = float("-inf")
    for x in arr:
        # TODO 1: cập nhật cur
        # TODO 2: i = vị trí của prefix nhỏ nhất >= cur - k
        # TODO 3: nếu i hợp lệ -> cập nhật best bằng cur - sorted_prefix[i]
        # TODO 4: chèn cur vào sorted_prefix (giữ thứ tự)
        pass
    return best

# Tự làm xong rồi mới mở test:
# assert exercise_max_subarray_at_most_k([5, -6, 3, -2, 4], 3) == 3
# assert exercise_max_subarray_at_most_k([2, -3, 4, -1, 3], 4) == 4
# assert exercise_max_subarray_at_most_k([-3], 0) == -3
# assert exercise_max_subarray_at_most_k([2, -1], 2) == 2

### Bài 4 — Tự viết lại: LeetCode 304 `NumMatrix` (Range Sum Query 2D - Immutable)

Đề (tóm tắt): thiết kế class `NumMatrix`:

- `NumMatrix(matrix)` — khởi tạo với ma trận `matrix`.
- `sumRegion(row1, col1, row2, col2)` — trả tổng hình chữ nhật góc trên-trái `(row1, col1)`, góc dưới-phải `(row2, col2)`. Mỗi lời gọi phải chạy `O(1)`.

Ví dụ của đề: với `matrix = [[3,0,1,4,2],[5,6,3,2,1],[1,2,0,1,5],[4,1,0,1,7],[1,0,3,0,5]]`: `sumRegion(2,1,4,3) = 8`, `sumRegion(1,1,2,2) = 11`, `sumRegion(1,2,2,4) = 12`.

Gợi ý mức 1: trong `__init__` xây bảng prefix 2D và lưu vào `self`.
Gợi ý mức 2: bảng kích thước `(m+1) x (n+1)`, `P[r+1][c+1] = P[r][c+1] + P[r+1][c] - P[r][c] + matrix[r][c]`.
Gợi ý mức 3: `sumRegion` = `P[r2+1][c2+1] - P[r1][c2+1] - P[r2+1][c1] + P[r1][c1]`.

In [34]:
class ExerciseNumMatrix:
    def __init__(self, matrix: List[List[int]]):
        # TODO 1: xây self.P kích thước (m+1) x (n+1)
        pass

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        # TODO 2: công thức bao hàm – loại trừ
        raise NotImplementedError

# Tự làm xong rồi mới mở test:
# nm = ExerciseNumMatrix([[3,0,1,4,2],[5,6,3,2,1],[1,2,0,1,5],[4,1,0,1,7],[1,0,3,0,5]])
# assert nm.sumRegion(2, 1, 4, 3) == 8
# assert nm.sumRegion(1, 1, 2, 2) == 11
# assert nm.sumRegion(1, 2, 2, 4) == 12

### Bài 5 — Biến thể: LeetCode 1074 (Number of Submatrices That Sum to Target)

Đề (tóm tắt): cho `matrix` và `target`, **đếm** số hình chữ nhật con có tổng **đúng bằng** `target`. Constraints: `1 <= matrix.length <= 100`, `1 <= matrix[0].length <= 100`, `-1000 <= matrix[i][j] <= 1000`, `-10^8 <= target <= 10^8`.

Ví dụ: `[[0,1,0],[1,1,1],[0,1,0]], target = 0 → 4`; `[[1,-1],[-1,1]], target = 0 → 5`; `[[904]], target = 0 → 0`.

Dùng **đúng khung nén cột** của bài này; chỉ thay bài 1D bằng "**đếm** số đoạn con có tổng bằng `target`".

Gợi ý mức 1: đoạn `arr[l..r]` có tổng `target` ⇔ `P[l] = cur - target`. Không cần sắp xếp — chỉ cần biết **bao nhiêu** prefix trước đó **bằng** `cur - target`.
Gợi ý mức 2: dùng `dict` đếm: `count = {0: 1}`; mỗi bước `total += count.get(cur - target, 0)` rồi `count[cur] = count.get(cur, 0) + 1`.
Gợi ý mức 3: cũng "hỏi trước, thêm sau" như bài 1D của ta; khởi tạo `{0: 1}` đóng vai `P[0]`.

In [35]:
def num_submatrix_sum_target(matrix, target):
    # TODO 1: với mỗi top: col_sum = [0] * n; với mỗi bottom >= top: cộng hàng bottom vào col_sum
    # TODO 2: đếm số đoạn con của col_sum có tổng == target bằng prefix sum + dict đếm
    # TODO 3: cộng dồn vào kết quả và return
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert num_submatrix_sum_target([[0, 1, 0], [1, 1, 1], [0, 1, 0]], 0) == 4
# assert num_submatrix_sum_target([[1, -1], [-1, 1]], 0) == 5
# assert num_submatrix_sum_target([[904]], 0) == 0

### Bài 6 — Mini challenge: tổng **nhỏ nhất** mà **không nhỏ hơn** `k`

Viết `min_sum_submatrix_at_least_k(matrix, k)`: trong các hình chữ nhật có tổng **`>= k`**, trả về tổng **nhỏ nhất** (giả sử luôn có ít nhất một hình như vậy). Đây là bài "soi gương" của #363 — buộc bạn phải tự suy lại chiều tìm kiếm.

Ví dụ: `[[1,0,1],[0,-2,3]], k = 2 → 2`; `[[1,0,1],[0,-2,3]], k = 4 → 4` (hàng 0..1, cột 2..2: `1 + 3`); `[[2,2,-1]], k = 0 → ?` — tự tính (gợi ý: các tổng của mọi đoạn là `2, 4, 3, 2, 1, -1`).

Gợi ý mức 1: bài 1D: `cur - P[l] >= k ⇔ P[l] <= cur - k`; muốn `cur - P[l]` **nhỏ nhất** ⇒ `P[l]` **lớn nhất** thỏa ⇒ tìm **floor** của `cur - k`.
Gợi ý mức 2: floor = `sorted_prefix[bisect.bisect_right(sorted_prefix, cur - k) - 1]`, hợp lệ khi chỉ số `>= 0`.
Gợi ý mức 3: khởi tạo `best = float("inf")`, dùng `min`; khung 2D (nén cột) giữ nguyên.

In [36]:
def min_sum_submatrix_at_least_k(matrix, k):
    # TODO 1: khung 2D: cố định top, bottom; nén cột vào col_sum
    # TODO 2: bài 1D: với mỗi cur, tìm prefix LỚN NHẤT <= cur - k (bisect_right - 1)
    # TODO 3: best = min(best, cur - floor); return best
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert min_sum_submatrix_at_least_k([[1, 0, 1], [0, -2, 3]], 2) == 2
# assert min_sum_submatrix_at_least_k([[1, 0, 1], [0, -2, 3]], 4) == 4
# assert min_sum_submatrix_at_least_k([[2, 2, -1]], 0) == ...   # tự điền đáp án

## 23. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

1. `2`, `4`, `0` (ceiling của `1` là `1` ở vị trí 2; `5` lớn hơn mọi phần tử ⇒ `len = 4`; `-3` nhỏ hơn mọi phần tử ⇒ `0`).
2. `[0, 3, 2, 4]`.
3. `[[0, 0, 0], [0, 1, 3], [0, 4, 10]]` (ví dụ `P[2][2] = P[1][2] + P[2][1] - P[1][1] + 4 = 3 + 4 - 1 + 4 = 10`).
4. `[[1, 4], [2, 5], [3, 6]]`.
5. `[3 + 5, -4 + 6] = [8, 2]`.

### Đáp án Bài 2

`arr = [2, -3, 4, -1, 3]`, `k = 4`, `P = [0, 2, -1, 3, 2, 5]`:

| `r` | `x` | `cur` | `need` | `sorted_prefix` trước | `i` | ceiling | `cur - ceiling` | đoạn | `best` |
|---:|---:|---:|---:|---|---:|---:|---:|---|---:|
| 0 | 2 | 2 | -2 | `[0]` | 0 | 0 | 2 | `arr[0..0]` | 2 |
| 1 | -3 | -1 | -5 | `[0, 2]` | 0 | 0 | -1 | `arr[0..1]` | 2 |
| 2 | 4 | 3 | -1 | `[-1, 0, 2]` | 0 | -1 | **4** | `arr[2..2]` | **4** |
| 3 | -1 | 2 | -2 | `[-1, 0, 2, 3]` | 0 | -1 | 3 | `arr[2..3]` | 4 |
| 4 | 3 | 5 | 1 | `[-1, 0, 2, 2, 3]` | 2 | 2 | 3 | `arr[1..4]` (hoặc `arr[4..4]`) | 4 |

Đáp án **`4`** (đoạn `[4]`, tổng đúng bằng `k`). Kadane = `6` (đoạn `[4, -1, 3]`) `> 4` ⇒ bộ lọc Kadane **không** dùng được, phải chạy bài 1D đầy đủ.

### Ghi chú Bài 5 và Bài 6

- **Bài 5:** khác #363 ở chỗ cần **đúng bằng** `target` và phải **đếm** ⇒ không cần thứ tự, chỉ cần `dict` đếm số lần mỗi prefix xuất hiện (`O(1)` trung bình mỗi bước) ⇒ `O(m²·n)`.
- **Bài 6:** `[[2,2,-1]], k = 0 → 1` (đoạn `[2, -1]`; các tổng `>= 0` là `2, 4, 3, 2, 1`). Chiều tìm kiếm đảo ngược: `cur - p >= k ⇔ p <= cur - k`, muốn tổng nhỏ nhất ⇒ `p` lớn nhất ⇒ **floor** = `bisect_right(...) - 1`.

Ô code dưới in lại trace Bài 2 để đối chiếu, rồi chạy đáp án Bài 3 → 6 kèm assert và test chéo với brute force.

In [37]:
# ---------- Bài 1 & Bài 2: kiểm chứng ----------
print([bisect.bisect_left([-2, 0, 1, 4], x) for x in (1, 5, -3)], prefix_1d([3, -1, 2]),
      build_prefix_2d([[1, 2], [3, 4]]), [list(c) for c in zip(*[[1, 2, 3], [4, 5, 6]])])
assert trace_subarray_at_most_k([2, -3, 4, -1, 3], 4) == 4
print("Kadane:", kadane([2, -3, 4, -1, 3]))

[2, 4, 0] [0, 3, 2, 4] [[0, 0, 0], [0, 1, 3], [0, 4, 10]] [[1, 4], [2, 5], [3, 6]]
r=0 x= 2 cur= 2 need=-2 sorted=[0] i=0 | ceiling =  0, tổng =  2, đoạn arr[0..0] | best=2
r=1 x=-3 cur=-1 need=-5 sorted=[0, 2] i=0 | ceiling =  0, tổng = -1, đoạn arr[0..1] | best=2
r=2 x= 4 cur= 3 need=-1 sorted=[-1, 0, 2] i=0 | ceiling = -1, tổng =  4, đoạn arr[2..2] | best=4
r=3 x=-1 cur= 2 need=-2 sorted=[-1, 0, 2, 3] i=0 | ceiling = -1, tổng =  3, đoạn arr[2..3] | best=4
r=4 x= 3 cur= 5 need= 1 sorted=[-1, 0, 2, 2, 3] i=2 | ceiling =  2, tổng =  3, đoạn arr[1..4] | best=4
Kadane: 6


In [38]:
# ---------- Bài 3 ----------
def answer_max_subarray_at_most_k(arr, k):
    sorted_prefix = [0]
    cur = 0
    best = float("-inf")
    for x in arr:
        cur += x
        i = bisect.bisect_left(sorted_prefix, cur - k)
        if i < len(sorted_prefix):
            best = max(best, cur - sorted_prefix[i])
        bisect.insort(sorted_prefix, cur)
    return best


# ---------- Bài 4 ----------
class AnswerNumMatrix:
    def __init__(self, matrix: List[List[int]]):
        m, n = len(matrix), len(matrix[0])
        self.P = [[0] * (n + 1) for _ in range(m + 1)]
        for r in range(m):
            for c in range(n):
                self.P[r + 1][c + 1] = self.P[r][c + 1] + self.P[r + 1][c] - self.P[r][c] + matrix[r][c]

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        P = self.P
        return P[row2 + 1][col2 + 1] - P[row1][col2 + 1] - P[row2 + 1][col1] + P[row1][col1]


# ---------- Bài 5 ----------
def answer_num_submatrix_sum_target(matrix, target):
    m, n = len(matrix), len(matrix[0])
    total = 0
    for top in range(m):
        col_sum = [0] * n
        for bottom in range(top, m):
            for c in range(n):
                col_sum[c] += matrix[bottom][c]
            count = {0: 1}                       # P[0] = 0 đã gặp 1 lần
            cur = 0
            for x in col_sum:
                cur += x
                total += count.get(cur - target, 0)   # hỏi trước
                count[cur] = count.get(cur, 0) + 1    # thêm sau
    return total


# ---------- Bài 6 ----------
def answer_min_sum_submatrix_at_least_k(matrix, k):
    m, n = len(matrix), len(matrix[0])
    best = float("inf")
    for top in range(m):
        col_sum = [0] * n
        for bottom in range(top, m):
            for c in range(n):
                col_sum[c] += matrix[bottom][c]
            sorted_prefix, cur = [0], 0
            for x in col_sum:
                cur += x
                i = bisect.bisect_right(sorted_prefix, cur - k) - 1   # prefix LỚN NHẤT <= cur - k
                if i >= 0:
                    best = min(best, cur - sorted_prefix[i])
                bisect.insort(sorted_prefix, cur)
    return best


assert answer_max_subarray_at_most_k([5, -6, 3, -2, 4], 3) == 3
assert answer_max_subarray_at_most_k([2, -3, 4, -1, 3], 4) == 4
assert answer_max_subarray_at_most_k([-3], 0) == -3
assert answer_max_subarray_at_most_k([2, -1], 2) == 2
nm = AnswerNumMatrix([[3, 0, 1, 4, 2], [5, 6, 3, 2, 1], [1, 2, 0, 1, 5], [4, 1, 0, 1, 7], [1, 0, 3, 0, 5]])
assert (nm.sumRegion(2, 1, 4, 3), nm.sumRegion(1, 1, 2, 2), nm.sumRegion(1, 2, 2, 4)) == (8, 11, 12)
assert answer_num_submatrix_sum_target([[0, 1, 0], [1, 1, 1], [0, 1, 0]], 0) == 4
assert answer_num_submatrix_sum_target([[1, -1], [-1, 1]], 0) == 5
assert answer_num_submatrix_sum_target([[904]], 0) == 0
assert answer_min_sum_submatrix_at_least_k([[1, 0, 1], [0, -2, 3]], 2) == 2
assert answer_min_sum_submatrix_at_least_k([[1, 0, 1], [0, -2, 3]], 4) == 4
assert answer_min_sum_submatrix_at_least_k([[2, 2, -1]], 0) == 1

# Test chéo với brute force trên input ngẫu nhiên
random.seed(1074)
for _ in range(800):
    m, n = random.randint(1, 4), random.randint(1, 4)
    mat = [[random.randint(-6, 6) for _ in range(n)] for _ in range(m)]
    rects = [(t, b, l, r) for t in range(m) for b in range(t, m) for l in range(n) for r in range(l, n)]
    sums = [sum(mat[i][j] for i in range(t, b + 1) for j in range(l, r + 1)) for t, b, l, r in rects]
    k = random.choice(sums) + random.randint(-2, 2)
    if min(mat[0]) <= k:                                   # hàng 0 có ít nhất một đoạn <= k
        assert answer_max_subarray_at_most_k(mat[0], k) == max_subarray_at_most_k_brute(mat[0], k)
    if any(s >= k for s in sums):
        assert answer_min_sum_submatrix_at_least_k(mat, k) == min(s for s in sums if s >= k)
    target = random.choice(sums)
    assert answer_num_submatrix_sum_target(mat, target) == sums.count(target)
    nmx = AnswerNumMatrix(mat)
    t, b, l, r = random.choice(rects)
    assert nmx.sumRegion(t, l, b, r) == sum(mat[i][j] for i in range(t, b + 1) for j in range(l, r + 1))
print("Đáp án Bài 3 → 6 PASS toàn bộ assert và 800 test chéo với brute force!")

Đáp án Bài 3 → 6 PASS toàn bộ assert và 800 test chéo với brute force!


## 24. Template cuối cùng nên nhớ

Ba câu thần chú: **"cố định cặp hàng, nén cột"** → **"bài 1D: tìm prefix nhỏ nhất `>= cur - k`"** → **"hỏi trước, thêm sau, bắt đầu từ `[0]`"**.

```python
class Solution:
    def maxSumSubmatrix(self, matrix: List[List[int]], k: int) -> int:
        m, n = len(matrix), len(matrix[0])
        if m > n:                                         # cặp trên chiều nhỏ
            matrix = [list(c) for c in zip(*matrix)]
            m, n = n, m
        best = float("-inf")
        for top in range(m):
            col_sum = [0] * n
            for bottom in range(top, m):
                for c in range(n):
                    col_sum[c] += matrix[bottom][c]       # nén cột
                sorted_prefix, cur = [0], 0               # P[0] = 0
                for x in col_sum:
                    cur += x
                    i = bisect.bisect_left(sorted_prefix, cur - k)
                    if i < len(sorted_prefix):
                        best = max(best, cur - sorted_prefix[i])
                    bisect.insort(sorted_prefix, cur)     # hỏi trước, thêm sau
                if best == k:
                    return k                              # dừng sớm
        return best
```

(Bản đầy đủ có thêm bộ lọc Kadane — mục 17.)

**Bản đồ tư duy:** tổng hình chữ nhật ⇒ prefix 2D ⇒ quá chậm vì `n²` cặp cột ⇒ cố định cặp hàng, nén cột ⇒ bài 1D có trần `k` + số âm ⇒ prefix + cấu trúc sắp xếp (ceiling).

## 25. Checklist trước khi submit

- [ ] Đúng tên method `maxSumSubmatrix(self, matrix, k)`, trả về `int`?
- [ ] `best` khởi tạo `float("-inf")` (đáp án có thể âm), **không** phải `0`?
- [ ] `sorted_prefix` bắt đầu bằng `[0]` để xét được đoạn bắt đầu từ cột `0`?
- [ ] Tìm **ceiling** bằng `bisect_left(sorted_prefix, cur - k)` (không phải `bisect_right`, không phải floor), và kiểm tra `i < len(...)` trước khi đọc?
- [ ] **Hỏi trước, `insort` sau** — không bao giờ ghép `cur` với chính nó (đoạn rỗng)?
- [ ] `col_sum` được tạo lại `[0] * n` cho mỗi `top`, và `bottom` chạy từ `top`?
- [ ] Nếu chuyển vị: đã đổi `m, n = n, m`?
- [ ] Bộ lọc Kadane đúng chiều: `kadane <= k` ⇒ lấy luôn; ngược lại chạy bài 1D?
- [ ] Đã thử: hai ví dụ của đề, ma trận 1 ô, `k` âm, mọi ô âm, ma trận 1 cột / 1 hàng, trường hợp phải "giữ" số lớn ở đầu (`[[4, -3, 2]], k = 3`)?
- [ ] Không dùng cửa sổ trượt (có số âm!)?